## Performance on describing final layer neurons of ResNet-50 (ImageNet)

In [1]:
import os
#virtually move to parent directory
os.chdir("..")

import torch
import pandas as pd
from sentence_transformers import SentenceTransformer

import clip
import utils
import data_utils
import similarity

/home/s4yadav/private/workspace/CLIP-dissect/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


## Arguments for CLIP-Dissect

In [2]:
clip_name = 'ViT-B/16'
target_name = 'resnet50'
target_layer = 'fc'
batch_size = 4
device = 'cuda'
pool_mode = 'avg'

save_dir = 'saved_activations'
similarity_fn = similarity.soft_wpmi

In [3]:
import quantization_utils_awq

# Quantization settings
quantize_enabled = True  # Set to False to disable quantization
quantization_bits = 8    # INT8 quantization
quantization_group_size = 0  # 0 = per-channel, otherwise group-wise

if quantize_enabled:
    target_model_to_use = quantization_utils_awq.quantize_given_mdodel(
        target_name = target_name,
        quantization_bits = quantization_bits,
        quantization_group_size = quantization_group_size,
        device = device,
    )
else:
    print(f"Quantization disabled. Using original {target_name} model.")
    target_model_to_use = None  # Will load default in save_activations

Loading resnet50 model for quantization...


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /tmp/xdg-cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 181MB/s]


Quantizing resnet50 with 8-bit AWQ quantization...
Quantized model saved to saved_activations/resnet50_quantized_8bit.pt
Model quantized successfully!
Model size statistics:
  INT8 parameters: 24.32 MB
  FP32 equivalent: 0.20 MB
  Compression ratio: 0.01x


In [4]:
# Create a simple wrapper for sentence embeddings using transformers directly
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

class SimpleSentenceTransformer:
    def __init__(self, model_name):
        print(f"Loading {model_name} using transformers...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        self.model.eval()
    
    def encode(self, texts):
        # Convert numpy arrays and other iterables to list of strings
        if isinstance(texts, np.ndarray):
            texts = texts.tolist()
        elif not isinstance(texts, list):
            texts = list(texts)
        if isinstance(texts, str):
            texts = [texts]
        
        # Ensure all elements are strings
        texts = [str(t) for t in texts]
        
        encoded = self.tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
        encoded = {k: v.to(self.device) for k, v in encoded.items()}
        with torch.no_grad():
            output = self.model(**encoded)
            # Use mean pooling
            embeddings = output.last_hidden_state.mean(dim=1)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        return embeddings.cpu().numpy()

# Load models
try:
    model = SimpleSentenceTransformer('sentence-transformers/all-mpnet-base-v2')
except Exception as e:
    print(f"Failed with all-mpnet-base-v2: {e}")
    print("Using distilbert as fallback...")
    model = SimpleSentenceTransformer('distilbert-base-uncased')

clip_model, _ = clip.load(clip_name, device=device)

with open('data/imagenet_labels.txt', 'r') as f: 
    imagenet_classnames = (f.read()).split('\n')

Loading sentence-transformers/all-mpnet-base-v2 using transformers...


tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [5]:
def save_activations_with_quantization(clip_name, target_name, target_model_to_use, target_layers, d_probe, concept_set, pool_mode, save_dir, batch_size, device):
    # Load models
    clip_model, clip_preprocess = clip.load(clip_name, device=device)

    if quantize_enabled and target_model_to_use is not None:
        # Use quantized model
        target_model = target_model_to_use
        target_preprocess = data_utils.get_target_model(target_name, device)[1]  # Get preprocessor only
    else:
        # Load default model
        target_model, target_preprocess = data_utils.get_target_model(target_name, device)

    # Setup data
    data_c = data_utils.get_data(d_probe, clip_preprocess)
    data_t = data_utils.get_data(d_probe, target_preprocess)

    with open(concept_set, 'r') as f: 
        words = (f.read()).split('\n')

    # Ignore empty lines
    words = [i for i in words if i != ""]

    # Generate text embeddings
    text = clip.tokenize(["{}".format(word) for word in words]).to(device)

    # Get save names
    save_names = utils.get_save_names(
        clip_name=clip_name,
        target_name=target_name,
        target_layer='{}',
        d_probe=d_probe,
        concept_set=concept_set,
        pool_mode=pool_mode,
        save_dir=save_dir
    )

    target_save_name, clip_save_name, text_save_name = save_names

    # Save features using quantized model (if enabled)
    print("Saving CLIP text features...")
    utils.save_clip_text_features(clip_model, text, text_save_name, batch_size)

    print("Saving CLIP image features...")
    utils.save_clip_image_features(clip_model, data_c, clip_save_name, batch_size, device)

    print(f"Saving target activations from {'quantized ' if quantize_enabled else ''}model...")
    utils.save_target_activations(
        target_model,
        data_t,
        target_save_name,
        target_layers,
        batch_size,
        device,
        pool_mode
    )

    print("Activation extraction complete!")


## Run CLIP-Dissect

In [6]:
rows = [("imagenet_val", "data/broden_labels_clean.txt"),
       ("imagenet_val", "data/3k.txt"),
       ("imagenet_val", "data/10k.txt"),
       ("imagenet_val", "data/20k.txt"),
       ("imagenet_val", "data/imagenet_labels.txt"),
       ("cifar100_train", "data/20k.txt"),
       ("broden", "data/20k.txt"),
       ("imagenet_val", "data/20k.txt"),
       ("imagenet_broden", "data/20k.txt"),]

In [7]:
for d_probe, concept_set in rows:
    with open(concept_set, 'r') as f: 
        words = (f.read()).split('\n')
    # utils.save_activations(clip_name = clip_name, target_name = target_name, target_layers = [target_layer], 
    #                        d_probe = d_probe, concept_set = concept_set, batch_size = batch_size, 
    #                        device = device, pool_mode=pool_mode, save_dir = save_dir)

    save_activations_with_quantization(clip_name = clip_name, 
                       target_name = target_name, 
                       target_model_to_use=target_model_to_use, 
                       target_layers = [target_layer], 
                       d_probe = d_probe, 
                       concept_set = concept_set, 
                       batch_size = batch_size, 
                       device = device, 
                       pool_mode=pool_mode, 
                       save_dir = save_dir
                    )

    save_names = utils.get_save_names(clip_name = clip_name, target_name = target_name,
                                      target_layer = target_layer, d_probe = d_probe,
                                      concept_set = concept_set, pool_mode=pool_mode,
                                      save_dir = save_dir)

    target_save_name, clip_save_name, text_save_name = save_names

    similarities, target_feats = utils.get_similarity_from_activations(target_save_name, clip_save_name, 
                                                        text_save_name, similarity_fn, device=device)

    clip_preds = torch.argmax(similarities, dim=1)
    clip_preds = [words[int(pred)] for pred in clip_preds]

    clip_cos, mpnet_cos = utils.get_cos_similarity(clip_preds, imagenet_classnames, clip_model, model, device, batch_size)
    print("D_probe:{}, Concept set:{}".format(d_probe, concept_set))
    print("CLIP-Dissect - Clip similarity: {:.4f}, mpnet similarity: {:.4f}".format(clip_cos, mpnet_cos))

Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


/home/s4yadav/private/workspace/CLIP-dissect/utils.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  image_features = torch.load(clip_save_name, map_location='cpu').floa

torch.Size([1000, 1197])
D_probe:imagenet_val, Concept set:data/broden_labels_clean.txt
CLIP-Dissect - Clip similarity: 0.7393, mpnet similarity: 0.5006
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


100%|██████████| 1000/1000 [00:02<00:00, 400.33it/s]


torch.Size([1000, 3000])
D_probe:imagenet_val, Concept set:data/3k.txt
CLIP-Dissect - Clip similarity: 0.7456, mpnet similarity: 0.4807
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


...

torch.Size([1000, 20000])
D_probe:broden, Concept set:data/20k.txt
CLIP-Dissect - Clip similarity: 0.7417, mpnet similarity: 0.4796
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


100%|██████████| 1000/1000 [00:06<00:00, 143.43it/s]


torch.Size([1000, 20000])
D_probe:imagenet_val, Concept set:data/20k.txt
CLIP-Dissect - Clip similarity: 0.7900, mpnet similarity: 0.5721
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


100%|██████████| 1000/1000 [00:07<00:00, 138.82it/s]


torch.Size([1000, 20000])
D_probe:imagenet_broden, Concept set:data/20k.txt
CLIP-Dissect - Clip similarity: 0.7900, mpnet similarity: 0.5713


## Baselines

In [8]:
netdissect_res = pd.read_csv('data/NetDissect_results/resnet50_imagenet_fc.csv')
nd_preds = netdissect_res['label'].values

clip_cos, mpnet_cos = utils.get_cos_similarity(nd_preds, imagenet_classnames, clip_model, model, device, batch_size)
print("Network Dissection - Clip similarity: {:.4f}, mpnet similarity: {:.4f}".format(clip_cos, mpnet_cos))

Network Dissection - Clip similarity: 0.6929, mpnet similarity: 0.3963


In [9]:
milan_preds = pd.read_csv('data/MILAN_results/m_base_resnet50_imagenet.csv')
milan_preds = milan_preds[milan_preds['layer']=='fc']
milan_preds = milan_preds.sort_values(by=['unit'])
milan_preds = list(milan_preds['description'])

clip_cos, mpnet_cos = utils.get_cos_similarity(milan_preds, imagenet_classnames, clip_model, model, device, batch_size)
print("MILAN - Clip similarity: {:.4f}, mpnet similarity: {:.4f}".format(clip_cos, mpnet_cos))

MILAN - Clip similarity: 0.7080, mpnet similarity: 0.3932
